In [2]:
# Load environment variables and configure the Groq chat model.
from pathlib import Path
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from dotenv import load_dotenv


env_path = Path.cwd() / ".env"
if not env_path.exists():
    env_path = Path.cwd().parent / ".env"
load_dotenv(env_path)  # Load environment variables from workspace root .env file


llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
    # other params...
)


# Output Parsers

Language models return raw text. **Output parsers** convert that text into structured data (strings, JSON, lists, Pydantic models, etc.).

Every output parser implements two key methods:
- `get_format_instructions()` — returns a string telling the LLM how to format its output.
- `parse(text)` — takes the LLM response string and converts it to a structured object.

Below are examples for the most commonly used parsers.

## 1. StrOutputParser

The simplest parser — just extracts the string content from the model's message output.
Useful at the end of a chain to get a plain string instead of an `AIMessage` object.

In [3]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# Create a simple prompt
prompt = ChatPromptTemplate.from_template("Tell me a one-line fun fact about {topic}.")

# Build a chain: prompt -> llm -> string parser
str_parser = StrOutputParser()
chain = prompt | llm | str_parser

# Invoke the chain — output is a plain Python string
result = chain.invoke({"topic": "octopuses"})
print(type(result))
print(result)

<class 'langchain_core.messages.base.TextAccessor'>
Octopuses have three hearts: two pump blood to the gills, and the third pumps to the rest of the body, which actually stops beating when they swim!


## 2. JsonOutputParser

Parses the LLM output into a Python `dict`. You can optionally pass a Pydantic schema
so that `get_format_instructions()` includes the expected JSON shape in the prompt.

In [5]:
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field


# Define the desired JSON shape using Pydantic
class Joke(BaseModel):
    setup: str = Field(description="The setup of the joke")
    punchline: str = Field(description="The punchline of the joke")


# Create the JSON parser tied to the Joke schema
json_parser = JsonOutputParser(pydantic_object=Joke)

# Inject format instructions into the prompt so the LLM knows the JSON shape
prompt = ChatPromptTemplate.from_template(
    "Tell me a joke about {topic}.\n{format_instructions}"
).partial(format_instructions=json_parser.get_format_instructions())

chain = prompt | llm | json_parser

# Result is a Python dict matching the Joke schema
result = chain.invoke({"topic": "programmers"})
print(type(result))
print(result)

<class 'dict'>
{'setup': 'Why do programmers prefer dark mode?', 'punchline': 'Because light attracts bugs!'}


## 3. CommaSeparatedListOutputParser (CSV)

Parses a comma-separated string from the LLM into a Python `list[str]`.
Great for "list 5 things..." style prompts.

In [6]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

# Create the CSV list parser
csv_parser = CommaSeparatedListOutputParser()

# Its format instructions tell the LLM to return a comma-separated list
prompt = ChatPromptTemplate.from_template(
    "List five {subject}.\n{format_instructions}"
).partial(format_instructions=csv_parser.get_format_instructions())

chain = prompt | llm | csv_parser

# Result is a Python list of strings
result = chain.invoke({"subject": "popular ice cream flavors"})
print(type(result))
print(result)

<class 'list'>
['vanilla', 'chocolate', 'strawberry', 'mint chocolate chip', 'cookies and cream']


## 4. DatetimeOutputParser

> **Note:** This parser is **not available in LangChain v1**. It existed in earlier versions
> (`langchain.output_parsers.DatetimeOutputParser`) and parsed model output into a
> `datetime.datetime` object. The snippet below is shown for reference only — in v1 you
> should use a `JsonOutputParser` or `PydanticOutputParser` with a `datetime` field instead.

```python
# Legacy usage (LangChain < v1):
# from langchain.output_parsers import DatetimeOutputParser
#
# dt_parser = DatetimeOutputParser()
# prompt = ChatPromptTemplate.from_template(
#     "When did the event '{event}' happen?\n{format_instructions}"
# ).partial(format_instructions=dt_parser.get_format_instructions())
#
# chain = prompt | llm | dt_parser
# result = chain.invoke({"event": "the moon landing"})
# print(type(result))  # datetime.datetime
# print(result)
```

## 5. Structured Output Parser (Pydantic)

`PydanticOutputParser` parses the LLM output directly into a **typed Pydantic model
instance** — giving you validation and attribute access (`obj.field`) instead of dict keys.

In [7]:
from langchain_core.output_parsers import PydanticOutputParser
from typing import List


# Define a richer schema
class Person(BaseModel):
    name: str = Field(description="Full name of the person")
    age: int = Field(description="Age in years")
    hobbies: List[str] = Field(description="List of the person's hobbies")


# Create the Pydantic parser
pydantic_parser = PydanticOutputParser(pydantic_object=Person)

prompt = ChatPromptTemplate.from_template(
    "Generate a fictional profile for a person named {name}.\n{format_instructions}"
).partial(format_instructions=pydantic_parser.get_format_instructions())

chain = prompt | llm | pydantic_parser

# Result is a validated Person instance — access fields by attribute
result = chain.invoke({"name": "Ada"})
print(type(result))
print(result)
print("Name:", result.name)
print("Age:", result.age)
print("Hobbies:", result.hobbies)

<class '__main__.Person'>
name='Ada Morgan' age=34 hobbies=['coding', 'hiking', 'photography']
Name: Ada Morgan
Age: 34
Hobbies: ['coding', 'hiking', 'photography']


### Bonus: Using `llm.with_structured_output()`

In LangChain v1, the recommended modern approach for structured output is
`llm.with_structured_output(Schema)`, which leverages the model's native
tool/function-calling instead of prompt-based parsing.

In [8]:
# Modern v1 alternative — no separate parser needed
structured_llm = llm.with_structured_output(Person)

result = structured_llm.invoke("Generate a fictional profile for a person named Grace.")
print(type(result))
print(result)

<class '__main__.Person'>
name='Grace' age=34 hobbies=['reading', 'hiking', 'painting']
